# BÀI TẬP: E-COMMERCE DATA (ONLINE RETAIL)
**Nguồn:** kaggle.com/datasets/carrie1/ecommerce-data (541,909 dòng)


## Setup

In [37]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from scipy import stats

sns.set_style('whitegrid')

csv_path = 'https://raw.githubusercontent.com/databricks/Spark-The-Definitive-Guide/master/data/retail-data/all/online-retail-dataset.csv'

df = pd.read_csv(csv_path, encoding='ISO-8859-1', on_bad_lines='skip')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print('Loaded from:', csv_path)
df.head()

Loaded from: https://raw.githubusercontent.com/databricks/Spark-The-Definitive-Guide/master/data/retail-data/all/online-retail-dataset.csv


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


---
# PHẦN A — DATA PROFILING
## A.1. Data size, column names, data types

In [38]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  str           
 1   StockCode    541909 non-null  str           
 2   Description  540455 non-null  str           
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), str(4)
memory usage: 33.1 MB


## A.2. Missing values & Duplicate data

In [39]:
print("Kiểm tra dữ liệu bị thiếu:")
print(df.isnull().sum())
print("\nKiểm tra dữ liệu trùng lặp:")
print(df.duplicated().sum())

Kiểm tra dữ liệu bị thiếu:


InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

Kiểm tra dữ liệu trùng lặp:
5268


## A.3. Invalid values

In [40]:
print("Quantity <= 0:", (df['Quantity'] <= 0).sum())
print("UnitPrice <= 0:", (df['UnitPrice'] <= 0).sum())

Quantity <= 0: 10624
UnitPrice <= 0: 2517


## A.4. Create a new column
Làm sạch dữ liệu (loại Quantity<=0, UnitPrice<=0), tạo cột `Sales` = Quantity * UnitPrice.

In [41]:
df_goc = df.copy()
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]
df['Sales'] = df['Quantity'] * df['UnitPrice']
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Sales
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34


---
# PHẦN B — DESCRIPTIVE STATISTICS
## Group 1 — Central Tendency

In [42]:
numeric_cols = ['Quantity', 'UnitPrice']
for col in numeric_cols:
    print(f"\n--- {col} ---")
    print("Mean: ", df[col].mean())
    print("Median: ", df[col].median())
    print("Mode: ", df[col].mode().tolist())



--- Quantity ---
Mean:  10.542037034242338
Median:  3.0
Mode:  [1]

--- UnitPrice ---
Mean:  3.9076252471213193
Median:  2.08
Mode:  [1.25]


## Group 2 — Dispersion

In [43]:
for col in numeric_cols:
    print(f"\n--- {col} ---")
    print("Range: ", df[col].max() - df[col].min())
    print("Variance:", df[col].var())
    print("Standard deviation:", df[col].std())
    print("IQR:", df[col].quantile(0.75) - df[col].quantile(0.25))


--- Quantity ---
Range:  80994
Variance: 24187.752993751637
Standard deviation: 155.52412351063626
IQR: 9.0

--- UnitPrice ---
Range:  13541.329
Variance: 1289.936149182571
Standard deviation: 35.915681104255434
IQR: 2.88


## Group 3 — Location and Shape

In [44]:
for col in numeric_cols:
    print(f"\n--- {col} ---")
    print("Min: ", df[col].min())
    print("Q1:", df[col].quantile(0.25))
    print("Median:", df[col].median())
    print("Q3:", df[col].quantile(0.75))
    print("Max:", df[col].max())
    print("Skewness:", df[col].skew())
    print("Kurtosis:", df[col].kurt())


--- Quantity ---
Min:  1
Q1: 1.0
Median: 3.0
Q3: 10.0
Max: 80995
Skewness: 471.72771633134437
Kurtosis: 236462.34282634684

--- UnitPrice ---
Min:  0.001
Q1: 1.25
Median: 2.08
Q3: 4.13
Max: 13541.33
Skewness: 206.08755495006466
Kurtosis: 62483.1427151048


---
# PHẦN C — DEFINE THE QUESTION

## Câu hỏi 1: Quốc gia nào đóng góp doanh thu cao nhất, chiếm bao nhiêu % tổng doanh thu?

In [45]:
df['doanh_thu'] = df['Quantity'] * df['UnitPrice']
quoc_gia = df.groupby('Country')['doanh_thu'].sum().sort_values(ascending = False)
print(f"Quốc gia đóng góp doanh thu cao nhất: {quoc_gia.index[0]}")
tong_doanh_thu = df['doanh_thu'].sum()
phan_tram = quoc_gia.iloc[0] / tong_doanh_thu * 100
print(f"Chiếm {phan_tram:.2f}% tổng doanh thu")

Quốc gia đóng góp doanh thu cao nhất: United Kingdom
Chiếm 84.61% tổng doanh thu


## Câu hỏi 2: Sản phẩm nào bán chạy nhất theo doanh thu?

In [46]:
sp_ban_chay = df.groupby('Description')['doanh_thu'].sum().sort_values(ascending = False)
print(f"Sản phẩm bán chạy nhất theo doanh thu là: {sp_ban_chay.index[0]}")


Sản phẩm bán chạy nhất theo doanh thu là: DOTCOM POSTAGE


## Câu hỏi 3: Doanh số có tính mùa vụ theo tháng không?

In [47]:
doanh_thu_thang = df.groupby(df['InvoiceDate'].dt.to_period('M'))['Sales'].sum()
print(doanh_thu_thang)

print(f"\nTháng có doanh thu cao nhất: {doanh_thu_thang.idxmax()}")
print(f"Doanh thu: {doanh_thu_thang.max():.2f}")


InvoiceDate
2010-12     823746.140
2011-01     691364.560
2011-02     523631.890
2011-03     717639.360
2011-04     537808.621
2011-05     770536.020
2011-06     761739.900
2011-07     719221.191
2011-08     759138.380
2011-09    1058590.172
2011-10    1154979.300
2011-11    1509496.330
2011-12     638792.680
Freq: M, Name: Sales, dtype: float64

Tháng có doanh thu cao nhất: 2011-11
Doanh thu: 1509496.33


## Câu hỏi 4: Giá trị đơn hàng trung bình (Average Order Value) khác nhau thế nào giữa các quốc gia?

In [48]:
tb_gtri = df.groupby(['Country', 'InvoiceNo'])['Sales'].sum().groupby('Country').mean().sort_values(ascending=False)

print(tb_gtri)

print(f"\nQuốc gia có giá trị đơn hàng trung bình cao nhất: {tb_gtri.index[0]}")
print(f"Giá trị đơn hàng trung bình cao nhất: {tb_gtri.iloc[0]:.2f}")


Country
Singapore               3039.898571
Netherlands             3036.663191
Australia               2430.198421
Japan                   1969.282632
Lebanon                 1693.880000
Hong Kong               1426.527273
Brazil                  1143.600000
Sweden                  1066.064722
Switzerland             1057.220370
Denmark                 1053.074444
Israel                  1016.907500
Norway                  1004.595556
RSA                     1002.310000
EIRE                     984.215139
Greece                   952.104000
Cyprus                   849.398750
Channel Islands          786.555385
USA                      716.078000
Spain                    684.190111
United Arab Emirates     634.093333
Iceland                  615.714286
Canada                   611.063333
Austria                  599.922353
Portugal                 581.846552
Finland                  549.904390
Malta                    545.118000
France                   534.987526
United Kingdom      

## Câu hỏi 5: Tỷ lệ giao dịch có dấu hiệu trả hàng/hủy (Quantity âm ở dữ liệu gốc) khác nhau thế nào giữa các quốc gia?

In [ ]:
ty_le_tra_hang = df_goc.groupby('Country')['Quantity'].apply(lambda x: (x < 0).mean() * 100).sort_values(ascending=False)
print(ty_le_tra_hang)

print(f"\nQuốc gia có tỷ lệ trả hàng/hủy cao nhất: {ty_le_tra_hang.index[0]}")
print(f"Tỷ lệ: {ty_le_tra_hang.iloc[0]:.2f}%")

Country
USA                     38.487973
Czech Republic          16.666667
Malta                   11.811024
Japan                   10.335196
Saudi Arabia            10.000000
Australia                5.877681
Italy                    5.603985
Bahrain                  5.263158
Germany                  4.770932
EIRE                     3.684724
Poland                   3.225806
Singapore                3.056769
Sweden                   2.380952
Denmark                  2.313625
Spain                    1.894986
United Kingdom           1.855178
Belgium                  1.836636
Switzerland              1.748252
France                   1.741264
European Community       1.639344
Finland                  1.438849
Hong Kong                1.388889
Channel Islands          1.319261
Norway                   1.289134
Cyprus                   1.286174
Portugal                 1.184990
Austria                  0.748130
Greece                   0.684932
Israel                   0.673401
Nether

## Câu hỏi 6 (Tổng hợp) — Viết insight tổng hợp
Dựa trên Phần A, B, C, viết 4-5 câu insight tổng thể.

United Kingdom là quốc gia đóng góp doanh thu cao nhất, chiếm 84,61% tổng doanh thu. Sản phẩm có doanh thu cao nhất là DOTCOM POSTAGE. Doanh thu có sự biến động theo tháng và đạt cao nhất vào tháng 11/2011. Singapore có giá trị đơn hàng trung bình cao nhất, khoảng 3039,90. USA có tỷ lệ giao dịch có dấu hiệu trả hàng/hủy cao nhất, khoảng 38,49%